# 00 — Teste de fumaça do acesso aos dados (Fase 0, passo 0.5)

Caderno temporário, **não faz parte da entrega**. Verifica se as fontes podem ser baixadas por código antes de investir na extração completa. Mostra apenas nomes de colunas e contagens — nenhum registro individual é impresso.

**Localização do repositório.** Garante que o caderno rode a partir de `notebooks/`, inclusive quando aberto isolado (ex.: Google Colab), caso em que o repositório público é clonado. Todos os caminhos seguintes são relativos à raiz.

In [ ]:
# Localiza a raiz do repositório (pasta com requirements.txt e src/appd).
# Se o caderno foi aberto isolado (ex.: Google Colab), clona o repositório
# público na pasta atual. Depois disso, tudo usa caminhos relativos à raiz.
import os
import subprocess
from pathlib import Path

URL_REPOSITORIO = "https://github.com/bsousa7/APPD_26.git"


def localizar_raiz():
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if (pasta / "requirements.txt").exists() and (pasta / "src" / "appd").exists():
            return pasta
    return None


if localizar_raiz() is None:
    if not Path("APPD_26").exists():
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, "APPD_26"], check=True)
    os.chdir("APPD_26")
os.chdir(localizar_raiz() / "notebooks")
print("Pasta de trabalho:", Path.cwd())

## 1. Ambiente
Instala dependências, define caminhos e aponta o cache do PySUS para `data/raw/pysus` (ignorado pelo Git).

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

RAIZ = Path("..")
sys.path.insert(0, str(RAIZ / "src"))

DIR_RAW = RAIZ / "data" / "raw"
DIR_PROC = RAIZ / "data" / "processed"
DIR_DIC = DIR_PROC / "dicionarios"
DIR_FIG = RAIZ / "outputs" / "figuras"
DIR_TAB = RAIZ / "outputs" / "tabelas"
for d in (DIR_RAW, DIR_PROC, DIR_DIC, DIR_FIG, DIR_TAB):
    d.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option("display.max_columns", 30)
print("Python", sys.version.split()[0], "| pandas", pd.__version__)

import pysus
from appd import download as dl
from appd.agregacao import TP_UNID_POLO

dl.configurar_cache(RAIZ)
print("PySUS", pysus.__version__)

O cache do PySUS passa a ficar dentro de `data/raw/`, que é esvaziado na entrega.

## 2. CNES-ST, DF, dezembro de 2012
**Por quê:** confirmar que o campo `TP_UNID` existe e que já há estabelecimentos do tipo 74 (Polo Academia da Saúde).

In [ ]:
st = dl.baixar_datasus("cnes", "DF", 2012, 12, grupo="ST")
print("Colunas:", list(st.columns))
print("Estabelecimentos tipo 74:", int((st["TP_UNID"].str.zfill(2) == TP_UNID_POLO).sum()))

**Resultado:** [PREENCHER após executar: as colunas esperadas aparecem? quantos polos tipo 74?]

## 3. SIH-RD, DF, janeiro de 2024
**Por quê:** confirmar os campos `MUNIC_RES`, `DIAG_PRINC` e `ANO_CMPT` e a disponibilidade de 2024.

In [ ]:
rd = dl.baixar_datasus("sih", "DF", 2024, 1, grupo="RD")
print("Colunas:", list(rd.columns)[:40], "...")
print("Campos necessários presentes:", {c: c in rd.columns for c in ["MUNIC_RES", "DIAG_PRINC", "ANO_CMPT", "IDADE", "COD_IDADE"]})
print("Linhas:", len(rd))

**Resultado:** [PREENCHER após executar]

## 4. IBGE: lista de municípios e população (SIDRA)
**Por quê:** confirmar a API de localidades e o número da tabela de estimativas populacionais (provavelmente 6579, variável 9324).

In [ ]:
mun = dl.municipios_ibge()
print("Municípios:", len(mun), "| UFs:", mun["uf"].nunique())
pop = dl.tabela_sidra("6579", "9324", "2015")
print("Colunas SIDRA:", list(pop.columns))
print("Linhas:", len(pop))

**Resultado:** [PREENCHER após executar]

## 5. Registro de acessos e versões

In [ ]:
display(dl.registro_acessos())
%pip show pysus differences linearmodels

Anote as versões e as datas de acesso no `README.md` e no registro de uso de IA.